# vjaudit pipeline notebook

This notebook runs the whole pipeline, from the dataset files to the final results table: load
questions, check videos, draw samples, extract frames, let video models answer, let judge models grade
the answers, export labelling batches for Claude, import the labels, and report.

Everything happens here, in plain Python cells you can read and change. The notebook does not import
the `vjaudit` package; it only reads the prompt texts from `../configs/prompts/`.

<a id="toc"></a>
**Table of contents**

- [0. How to use this notebook](#s0)
- [1. Prerequisites: install and download](#s1)
- [2. Settings](#s2)
- [3. Helpers](#s3)
- [4. Load the benchmarks](#s4)
- [5. Check the videos](#s5)
- [6. Draw the samples](#s6)
- [7. Frames](#s7)
- [8. Generate answers](#s8)
- [9. Judge the answers](#s9)
- [10. Label with Claude](#s10)
- [11. Report](#s11)
- [12. Troubleshooting and next steps](#s12)

<a id="s0"></a>
## 0. How to use this notebook

**The idea in two sentences.** A *judge* model grades free-text answers of video models against the
benchmark's reference answer. This project measures how much those grades can be trusted, and whether
the judge treats some models (for example ones answering *without* the video) more kindly than others.
`../docs/EXPLAINED.md` explains it at length.

**Running it.**

1. Open the notebook in JupyterLab or VS Code with the `dl` environment as the kernel.
2. Set your choices in section 2 (normally only `PROFILE` and `RUN`).
3. Run the cells from top to bottom (JupyterLab: *Run → Run All Cells*), or one section at a time.

**Mac or server.** One switch in section 2, `PROFILE = "mac"` or `"server"`, chooses the models, judges,
sample sizes and judge stages. Every section ends its explanation with a **Mac vs server** line that
says what (if anything) differs.

**Two runs.**

| Run | Setting | Questions per benchmark (Mac) | What happens |
|---|---|---|---|
| Run 1 | `RUN = "annotated"` | 100 | answers, judging, Claude labels, report |
| Run 2 | `RUN = "main"` | 1,000 | more answers and judging with the main judge; Run 1's work is reused |

**Stopping and resuming.** Every long step saves its results as it goes. If the kernel stops, the laptop
sleeps or you press *Stop*, run the same section again: finished work is skipped. After a kernel
restart, run sections 2 and 3 first (settings and helpers), then jump to any section; each section reloads
what it needs from disk and tells you which earlier section to run if something is missing.

**Try it first in a dry run.** With `DRY_RUN = True` the notebook uses 3 questions per benchmark and
stand-in models that load nothing. It runs end to end in about a minute and checks that your data and
folders are set up. Its results go to a separate folder.

**Where results go.** Everything is written to `notebooks/runs/<profile>/` (git ignores it):

| File or folder | Contents |
|---|---|
| `pool.jsonl` | every question of the three benchmarks, with its video file and whether it decodes |
| `items.jsonl` | the sampled questions and the sets they belong to |
| `generations.jsonl` | every answer of every system |
| `judgecalls.jsonl` | every judge verdict, margin and raw reply |
| `units.jsonl`, `labels.jsonl` | the distinct answers given to Claude; the imported labels |
| `frames/`, `judge_cache/` | cached video frames and judge replies |
| `annotation/claude/` | batch files for Claude, and a `replies/` folder for Claude's answers |
| `results.md` | the final report |

**Long runs on a Mac.** Keep the laptop plugged in. To stop it from sleeping, run `caffeinate -dimsu` in a
terminal while the notebook works.

<a id="s1"></a>
## 1. Prerequisites: install and download

Do these once. They are **commented out** so that *Run All* never reinstalls packages or starts a
130 GB download by accident. To run one, remove the `#` in front of the command and run the cell, or
paste the command (without the leading `!`) into a terminal in the `vjauditv2` folder.

**1a. Packages.** Install them into the environment this notebook's kernel uses (`dl`):

- on the Mac: `torch`, `transformers` and helpers (`mac` extra), plus Jupyter, progress bars and plots
  (`notebook` extra);
- on the GPU server: `vllm` instead (`gpu` extra);
- the OpenAI client (`openai`) is already part of the base install; it is only needed for the optional
  GPT-3.5 stages.

In [ ]:
# Mac (run once, in the dl environment):
# !pip install -e "..[mac,notebook]"
#
# GPU server:
# !pip install -e "..[gpu,notebook]"

**1b. Datasets.** The download scripts fetch the three benchmarks (questions and videos) into
`vjauditv2/data/` by default. You already downloaded them into `vjaudit/data/`, and section 2 points
there on the Mac, so **skip this on your Mac**. On a new machine, run one of these:

| Script | What it fetches | Size |
|---|---|---|
| `download_all.sh` | all three benchmarks, and creates `configs/local/paths.yaml` | about 135 GB |
| `download_msvd.sh` | MSVD-QA questions (MEGA) and the 1,970 YouTube clips | about 3.5 GB |
| `download_anetqa.sh` | ActivityNet-QA questions (GitHub) and 5,044 videos (Hugging Face) | about 125 GB |
| `download_nextqa.sh` | NExT-QA open-ended and multiple-choice questions, and 1,570 videos | about 6 GB |

Each script accepts `--annotations-only` to fetch only the question files (a few MB), which is enough to
look at the data in section 4 before the videos arrive. MSVD needs `megatools`
(`brew install megatools` on a Mac).

In [ ]:
# Everything (questions + videos), into ../data:
# !bash ../scripts/download/download_all.sh
#
# Or one benchmark at a time:
# !bash ../scripts/download/download_msvd.sh
# !bash ../scripts/download/download_anetqa.sh
# !bash ../scripts/download/download_nextqa.sh
#
# Question files only (fast), for a first look:
# !bash ../scripts/download/download_all.sh --annotations-only

**1c. Judge prompt check (optional, once).** The T-VCG judge prompt should be word for word the
official Video-ChatGPT prompt. This script fetches the official one and compares. Without it, the
notebook still runs; the report then notes that the prompts were not verified.

In [ ]:
# !bash ../scripts/download/fetch_upstream_prompts.sh

<a id="s2"></a>
## 2. Settings

**The next cell is the only one you normally edit.** Run it, and the cell after it, every time you
start the notebook.

| Setting | Mac | Server | Meaning |
|---|---|---|---|
| `PROFILE` | `"mac"` | `"server"` | which models, judges, sample sizes and stages to use |
| `DRY_RUN` | `False` (`True` for a 1-minute check) | same | 3 questions per benchmark, stand-in models, separate results folder |
| `DATA_ROOT` | `None` → `../../vjaudit/data` | `None` → `../data` | folder holding `msvd/`, `anetqa/`, `nextqa/` |
| `RUN` | `"annotated"` (Run 1), later `"main"` (Run 2) | same | which sample the answer and judge steps work on |
| `RUN_LIMIT` | `None`, or `5` for a first real test | same | at most this many questions per benchmark in sections 8–9 |
| `RUN_STAGES` | `None` | `None` | judge stages to run; `None` = all non-optional stages of this run |
| `REBUILD_POOL` / `REBUILD_SAMPLE` | `False` | `False` | `True` redoes section 5 / 6 from scratch (old tables are kept with a timestamp) |

**Mac vs server:** change `PROFILE`. On the server, also make sure `OPENAI_API_KEY` is set in the
environment before starting Jupyter (the GPT-3.5 judge needs it).

In [ ]:
# ── The settings you normally change ─────────────────────────────────────────────────────────────
PROFILE = "mac"          # "mac" on the MacBook · "server" on the GPU server
DRY_RUN = False          # True = 3 questions per benchmark + stand-in models, to check the notebook in ~1 minute
DATA_ROOT = None         # None = Mac: ../../vjaudit/data (v1's data folder) · server: ../data · or a path string
RUN = "annotated"        # "annotated" = Run 1 · "main" = Run 2 (reuses everything Run 1 did)
RUN_LIMIT = None         # None = all questions · 5 = a quick first real test (5 questions per benchmark)
RUN_STAGES = None        # None = every non-optional judge stage of this run · e.g. ["M1", "M2"] or ["G1", "G2"]
REBUILD_POOL = False     # True = redo section 5 (video check) from scratch
REBUILD_SAMPLE = False   # True = redraw the samples in section 6 (do this only before generating answers)

The next cell loads the Python libraries the notebook uses. `tqdm` (progress bars) is optional.

In [ ]:
import ast
import copy
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import string
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:  # progress bars are optional
    def tqdm(iterable=None, **kwargs):
        return iterable

pd.set_option("display.max_colwidth", 80)

try:
    display  # provided by Jupyter
except NameError:  # plain Python (tests): print instead
    def display(obj):
        print(obj)

The next cell lists the models, judges and judge stages of each profile. Edit it only to change the set-up.

- **Generators** (video models). `styles` adds one system per answer style (`short` = one word or phrase,
  `sentence` = one sentence plus a reason). `blind: True` adds a system that answers *without* the video.
  `frame_ablation` adds systems that see only k frames. `revision: None` uses the latest version on the
  Hugging Face Hub; put a commit id there to pin it (section 8 prints the ids it used).
- **Judges.** `transformers` judges run locally; `openai` judges call the API (only the GPT-3.5 entries).
- **Stages.** One block of judging: which set, which judge, which prompt template, and the decoding.
  `optional: True` stages run only when named in `RUN_STAGES`.

**Mac vs server:** nothing to edit here; `PROFILE` picks `MAC` or `SERVER`.

In [ ]:
T0_VCG = dict(temperature=0, max_tokens=64, logprobs=True, top_logprobs=20, n=1)  # greedy, T-VCG reply
T0_RUB = dict(temperature=0, max_tokens=1, logprobs=True, top_logprobs=20, n=1)   # greedy, one-word reply


def _stage(name, judge, subset, template, request, systems="core", decoding="t0", optional=False):
    return dict(name=name, judge_id=judge, subset=subset, systems=systems, template=template,
                decoding_tag=decoding, request=request, optional=optional)


GPT35 = [
    dict(id="gpt-3.5-turbo-0125", backend="openai", model="gpt-3.5-turbo-0125", max_usd=100,
         price_in_per_mtok=0.50, price_out_per_mtok=1.50),
    dict(id="gpt-3.5-turbo-1106", backend="openai", model="gpt-3.5-turbo-1106", max_usd=20,
         price_in_per_mtok=0.50, price_out_per_mtok=1.50),
]

MAC = dict(
    device="mps",
    generators=[
        dict(id="qwen25vl3b", family="qwen-vl", repo="Qwen/Qwen2.5-VL-3B-Instruct", revision=None,
             backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short", "sentence"],
             blind=True, frame_ablation=[]),
        dict(id="qwen3vl2b", family="qwen-vl", repo="Qwen/Qwen3-VL-2B-Instruct", revision=None,
             backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short"],
             blind=False, frame_ablation=[]),
        dict(id="llavaov05b", family="llava-onevision", repo="llava-hf/llava-onevision-qwen2-0.5b-ov-hf",
             revision=None, backend="transformers", dtype="auto", n_frames={"default": 8}, styles=["short"],
             blind=False, frame_ablation=[]),
    ],
    frame_max_side=448,
    judges=[
        dict(id="qwen3-4b", backend="transformers", model="Qwen/Qwen3-4B-Instruct-2507", revision=None),
        dict(id="qwen25-1.5b", backend="transformers", model="Qwen/Qwen2.5-1.5B-Instruct", revision=None),
        GPT35[0],
    ],
    samples=dict(main=1000, annotated=100, vid=30),
    stages=[
        _stage("M1", "qwen3-4b", "annotated", "t-vcg", T0_VCG),
        _stage("M2", "qwen3-4b", "annotated", "t-rub", T0_RUB),
        _stage("M3", "qwen25-1.5b", "annotated", "t-vcg", T0_VCG),
        _stage("M4", "qwen25-1.5b", "annotated", "t-rub", T0_RUB),
        _stage("M5", "qwen3-4b", "main", "t-vcg", T0_VCG),
        _stage("M6", "qwen3-4b", "main", "t-rub", T0_RUB),
        _stage("G1", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG, optional=True),
        _stage("G2", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB, optional=True),
    ],
)

_SERVER_FRAMES = {"default": 16, "anetqa": 32}
SERVER = dict(
    device="cuda",
    generators=[
        dict(id="qwen25vl7b", family="qwen-vl", repo="Qwen/Qwen2.5-VL-7B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=True,
             frame_ablation=[]),
        dict(id="qwen3vl8b", family="qwen-vl", repo="Qwen/Qwen3-VL-8B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short", "sentence"],
             blind=True, frame_ablation=[1, 4]),
        dict(id="qwen3vl2b", family="qwen-vl", repo="Qwen/Qwen3-VL-2B-Instruct", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=False,
             frame_ablation=[]),
        dict(id="internvl35_8b", family="internvl", repo="OpenGVLab/InternVL3_5-8B", revision=None, backend="vllm",
             dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=32768, styles=["short"], blind=True,
             frame_ablation=[]),
        dict(id="llavanextvideo7b", family="llava-next-video", repo="llava-hf/LLaVA-NeXT-Video-7B-hf", revision=None,
             backend="vllm", dtype="bfloat16", n_frames=_SERVER_FRAMES, max_model_len=8192, styles=["short"],
             blind=False, frame_ablation=[]),
    ],
    frame_max_side=768,
    judges=GPT35,
    samples=dict(main=2000, annotated=300, vid=200),
    stages=[
        _stage("S1", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG),
        _stage("S2", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB),
        _stage("S3", "gpt-3.5-turbo-0125", "main", "t-vcg", T0_VCG),
        _stage("S4", "gpt-3.5-turbo-0125", "main", "t-rub", T0_RUB),
        _stage("S5", "gpt-3.5-turbo-0125", "annotated", "t-vcg",
               dict(temperature=None, max_tokens=64, logprobs=True, top_logprobs=20, n=3), decoding="default-n3"),
        _stage("S6a", "gpt-3.5-turbo-0125", "annotated", "t-vcg", T0_VCG, systems="ablation"),
        _stage("S6b", "gpt-3.5-turbo-0125", "annotated", "t-rub", T0_RUB, systems="ablation"),
        _stage("S6c", "gpt-3.5-turbo-0125", "main", "t-vcg", T0_VCG, systems="ablation"),
        _stage("S6d", "gpt-3.5-turbo-0125", "main", "t-rub", T0_RUB, systems="ablation"),
        _stage("S7", "gpt-3.5-turbo-1106", "annotated", "t-vcg", T0_VCG),
    ],
)

SHARED = dict(seed=42, msvd_min_per=15, frac_dev=0.4, max_new_tokens={"short": 128, "sentence": 256, "mc": 8})
BENCHMARKS = ["msvd", "anetqa", "nextoe"]


def resolve_settings(params: dict, repo: Path) -> dict:
    """Combine the switches above with the chosen profile into one settings dict ``S``."""
    profile = params["PROFILE"]
    if profile not in ("mac", "server"):
        raise ValueError(f'PROFILE must be "mac" or "server", not {profile!r}')
    if params.get("RUN", "annotated") not in ("annotated", "main"):
        raise ValueError('RUN must be "annotated" or "main"')
    base = copy.deepcopy(params.get("MAC", MAC) if profile == "mac" else params.get("SERVER", SERVER))
    dry = bool(params.get("DRY_RUN"))
    run = params.get("RUN", "annotated")
    if params.get("DATA_ROOT"):
        data_root = Path(params["DATA_ROOT"]).expanduser()
    else:
        data_root = repo.parent / "vjaudit" / "data" if profile == "mac" else repo / "data"
    if dry:
        base["samples"] = dict(main=3, annotated=3, vid=1)
        for g in base["generators"]:
            g["backend"] = "fake"
        for j in base["judges"]:
            j["backend"] = "fake"
    wanted = params.get("RUN_STAGES")
    if wanted:
        unknown = sorted(set(wanted) - {st["name"] for st in base["stages"]})
        if unknown:
            raise ValueError(f"unknown stage(s) {unknown}; stages are {[st['name'] for st in base['stages']]}")
        stages = [st for st in base["stages"] if st["name"] in wanted]
    else:
        stages = [st for st in base["stages"] if st["subset"] == run and not st["optional"]]
    return dict(SHARED, profile=profile, dry_run=dry, run=run, run_limit=params.get("RUN_LIMIT"),
                data_root=data_root, repo=repo, prompts=repo / "configs" / "prompts",
                runs=repo / "notebooks" / "runs" / (profile + ("-dryrun" if dry else "")),
                device=base["device"], generators=base["generators"], judges=base["judges"],
                frame_max_side=base["frame_max_side"], samples=base["samples"], stages=stages,
                all_stages=base["stages"], rebuild_pool=bool(params.get("REBUILD_POOL")),
                rebuild_sample=bool(params.get("REBUILD_SAMPLE")))

def detect_device(preferred: str) -> str:
    """The device that will really be used: the preferred one if torch can use it, else cpu."""
    try:
        import torch
    except ImportError:
        return "cpu (torch not installed)"
    if preferred == "mps" and torch.backends.mps.is_available():
        return "mps"
    if preferred == "cuda" and torch.cuda.is_available():
        return "cuda"
    return "cpu"


def show_settings(S: dict) -> None:
    """Print the resolved settings, the device and the main package versions."""
    rows = [("profile", S["profile"] + ("  (DRY RUN)" if S["dry_run"] else "")), ("run", S["run"]),
            ("run_limit", S["run_limit"]), ("data_root", S["data_root"]),
            ("data_root exists", S["data_root"].exists()), ("results folder", S["runs"]),
            ("samples (main/annotated/vid)", "/".join(str(S["samples"][k]) for k in ("main", "annotated", "vid"))),
            ("generators", ", ".join(f'{g["id"]} [{g["backend"]}]' for g in S["generators"])),
            ("judges", ", ".join(f'{j["id"]} [{j["backend"]}]' for j in S["judges"])),
            ("judge stages this run", ", ".join(st["name"] for st in S["stages"]) or "none"),
            ("frames", f'{S["generators"][0]["n_frames"]} at max {S["frame_max_side"]} px'),
            ("device", "none needed (dry run)" if S["dry_run"] else detect_device(S["device"]))]
    for mod in ("torch", "transformers", "vllm", "openai", "av"):
        try:
            rows.append((mod, __import__(mod).__version__))
        except Exception:  # noqa: BLE001 — not installed is fine for packages this profile does not use
            rows.append((mod, "not installed"))
    display(pd.DataFrame(rows, columns=["setting", "value"]).set_index("setting"))

The next cell applies the settings and prints them, with the versions of the main packages and the
compute device. Check that `data_root` points at your data and that the device is `mps` on the Mac.

In [ ]:
REPO = Path(os.environ.get("NOTEBOOK_REPO", Path.cwd().parent)).resolve()
S = resolve_settings(dict(PROFILE=PROFILE, DRY_RUN=DRY_RUN, DATA_ROOT=DATA_ROOT, RUN=RUN, RUN_LIMIT=RUN_LIMIT,
                          RUN_STAGES=RUN_STAGES, REBUILD_POOL=REBUILD_POOL, REBUILD_SAMPLE=REBUILD_SAMPLE), REPO)
S["runs"].mkdir(parents=True, exist_ok=True)
show_settings(S)

<a id="s3"></a>
## 3. Helpers

**What.** A few small tools used by every later section:

- **`Store`** saves results as tables of JSON lines, one file per table (`items.jsonl`, `generations.jsonl`,
  …). Rows are only ever appended, each with a key; a key that is already stored is skipped. This is what
  makes every step safe to stop and re-run.
- **`short_id`** turns a list of strings into a 16-character ID (SHA-1). A generation's ID is built from the
  question, the system and its full configuration, so changing a model or a frame setting makes new rows
  instead of overwriting old ones.
- **`normalize_answer`** puts an answer in a standard form (lower case, no punctuation, no leading
  "a/an/the"), so that "A dog." and "dog" count as the same answer.
- **`need`** stops a section with a clear message when an earlier section has not been run.

**Why.** Long runs get interrupted. With these, no result is lost or duplicated.

**Reads → writes.** Nothing yet; this section only defines tools.

**Mac vs server:** nothing to change.

In [ ]:
class Store:
    """Append-only tables of JSON lines in one folder; one row per key (the first one wins)."""

    def __init__(self, root: Path):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self._keys: dict[str, set] = {}

    def path(self, table: str) -> Path:
        return self.root / f"{table}.jsonl"

    def _rows(self, table: str) -> list[dict]:
        p = self.path(table)
        if not p.exists():
            return []
        rows = []
        lines = p.read_text(encoding="utf-8").splitlines()
        for i, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i == len(lines) - 1:  # a crash while writing the last line: ignore that line
                    continue
                raise
        return rows

    def load(self, table: str) -> pd.DataFrame:
        """The table as a DataFrame (empty if it does not exist yet), keeping the first row per key."""
        rows, seen, out = self._rows(table), set(), []
        for r in rows:
            k = r.get("_key")
            if k in seen:
                continue
            seen.add(k)
            out.append({c: v for c, v in r.items() if c != "_key"})
        return pd.DataFrame(out)

    def keys(self, table: str) -> set:
        if table not in self._keys:
            self._keys[table] = {r.get("_key") for r in self._rows(table)}
        return self._keys[table]

    def append(self, table: str, rows: list[dict], key: str) -> int:
        """Append rows whose `key` column is new; return how many were written."""
        known = self.keys(table)
        new = [r for r in rows if str(r[key]) not in known]
        if not new:
            return 0
        with open(self.path(table), "a", encoding="utf-8") as f:
            for r in new:
                f.write(json.dumps({"_key": str(r[key]), **r}, ensure_ascii=False, default=_json_default) + "\n")
                known.add(str(r[key]))
            f.flush()
            os.fsync(f.fileno())
        return len(new)

    def rebuild(self, table: str) -> Path | None:
        """Rename the table out of the way (with a timestamp) so the section can rebuild it."""
        p = self.path(table)
        self._keys.pop(table, None)
        if not p.exists():
            return None
        old = p.with_name(f"{table}.{datetime.now():%Y%m%d-%H%M%S}.old.jsonl")
        p.rename(old)
        return old


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"cannot save {type(o).__name__}")


def short_id(*parts: str) -> str:
    """16-hex SHA-1 of the parts joined by '|'."""
    return hashlib.sha1("|".join(parts).encode("utf-8")).hexdigest()[:16]


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False, default=_json_default)


_PUNCT = re.compile("[" + re.escape(string.punctuation) + "]")
_ARTICLE = re.compile(r"^(a|an|the) ")


def normalize_answer(text: str) -> str:
    """NFKC, lower case, punctuation → spaces, single spaces, one leading article removed.
    A non-empty answer never becomes empty."""
    s = unicodedata.normalize("NFKC", text).lower()
    out = " ".join(_PUNCT.sub(" ", s).split())
    out = _ARTICLE.sub("", out, count=1)
    if not out and text.strip():
        return " ".join(s.split())
    return out


def need(df: pd.DataFrame, section: int, what: str) -> pd.DataFrame:
    """Stop with a clear message when an earlier section's output is missing."""
    if df is None or len(df) == 0:
        raise RuntimeError(f"No {what} yet: run section {section} first (and sections 2–3 after a restart).")
    return df

Open the results store for this profile.

In [ ]:
store = Store(S["runs"])
print("results folder:", store.root)
print("tables so far:", sorted(p.stem for p in store.root.glob("*.jsonl")) or "none")